# MoE experiments with several seeds

A companion to `moe_upcycling.ipynb`. That notebook ran each experiment **once**, so differences under about 0.005 couldn't be told apart from luck. This one repeats every experiment with **3 seeds** and tests which differences are real.

**How it works**

* **Same starting point.** Every run converts the *same* trained dense model: `dense.pt` from the GitHub repo (validation loss 0.7848). If the download fails, the notebook trains a new one (about 3 minutes).
* **What a seed changes.** The seed decides which neurons are redrawn, the router's starting weights, the probabilistic routing samples, and the order of training batches.
* **Paired comparisons.** Within one seed, every variant and the dense control see the **same training batches**. So "variant minus baseline" can be compared seed by seed, which cancels much of the noise.
* **The test.** A paired t-test over the seeds. A difference counts as **real** if p < 0.05; with 3 seeds that means the average difference must be about 2.5× larger than the spread of the per-seed differences.
* **Resumable.** Results are saved after every run (to Google Drive if you allow it). If Colab disconnects, run all cells again: finished runs are skipped.

**Time:** 9 variants + 1 dense control, × 3 seeds, × 1,000 steps each ≈ **75–90 minutes** on a T4. To shorten it, cut `RUN` or `SEEDS` in section 8.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Run all.

In [ ]:
import os, math, time, copy, urllib.request
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU: switch the runtime to T4)")

## 1. Configuration
Everything you might want to experiment with lives here.

In [ ]:
QUICK = os.environ.get("QUICK") == "1"   # tiny smoke-test run; leave off in Colab

# --- data ---
dataset = "tinystories"       # "tinystories" (22.5M chars) | "shakespeare" (1.1M chars, overfits after ~1 epoch)

# --- dense ("linear") model ---
block_size, batch_size = 128, 64
n_layer, n_head, n_embd = 4, 4, 256
ffn_mult = 3                  # SwiGLU inner width = 3 * n_embd = 768 (Qwen3 uses 6,144 for 2,048)
dropout = 0.1
dense_iters, lr_dense = 2500, 1e-3

# --- growing into an MoE [§15] ---
n_experts, top_k = 8, 2       # routed experts per layer, experts per token
use_shared = True             # shared expert = first half of the dense neurons; routed experts come from the other half
init_method = "drop"          # "copy" | "partition" | "drop"
drop_ratio = 0.5              # drop: fraction of each expert's neurons redrawn (r = 0.5 was best)
zero_init_dropped = True      # drop: redrawn neurons start with zero down-projection -> smaller bump
part_frac = 0.5               # partition: each expert takes an overlapping random slice of this fraction of neurons
copy_noise = 0.0              # copy: optional relative noise; 0 keeps conversion exact

# --- router [§7] ---
score_fn = "sigmoid"          # "sigmoid" | "softmax"
router_init_std = 0.002       # one tenth of the usual 0.02
prob_topk_iters = 200         # probabilistic top-k for the first N steps, then hard top-k

# --- balancing [§12-§14] ---
balance = "lossfree"          # "lossfree" | "aux" | "none"
bias_gamma = 1e-3             # loss-free bias update speed
seq_aux_alpha = 1e-4          # tiny sequence-level loss kept alongside the bias (DeepSeek-V3 recipe); 0 to disable
aux_alpha = 1e-2              # Switch-style batch loss, only when balance == "aux"

# --- continued training (dense and MoE branches get the same budget) ---
cont_iters, lr_cont, warmup = 2000, 3e-4, 100
eval_every, eval_batches = 100, 20

if QUICK:
    batch_size, n_embd, n_head = 16, 64, 4
    dense_iters, cont_iters, eval_every, eval_batches, prob_topk_iters, warmup = 60, 60, 20, 4, 20, 10

## 2. Data (character level)
TinyStories by default. Point `DATASETS` at any text file you like; the rest of the notebook doesn't care.

In [ ]:
from collections import Counter
DATASETS = {
    "shakespeare": ("input.txt", "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"),
    "tinystories": ("tinystories_valid.txt", "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt"),
}
fname, url = DATASETS[dataset]
if not os.path.exists(fname):
    print("downloading", url)
    urllib.request.urlretrieve(url, fname)
text = open(fname, encoding="utf-8").read().replace("<|endoftext|>", "\n")
# fold very rare characters (stray unicode) into "?" so the vocabulary stays small
counts = Counter(text)
rare = {c for c, k in counts.items() if k < 50}
if rare:
    text = text.translate({ord(c): "?" for c in rare})
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
print(f"{dataset}: {len(text):,} characters, vocab {vocab_size}, "
      f"{dense_iters * batch_size * block_size / len(train_data):.1f} passes over the training text in the dense stage")

def get_batch(split, gen=None):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,), generator=gen)
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

# fixed validation batches, so "loss before" and "loss after" conversion are measured on identical data
_g = torch.Generator().manual_seed(0)
val_set = [get_batch("val", _g) for _ in range(eval_batches)]

## 3. The dense model

A small GPT whose feed-forward block is a SwiGLU network, the same shape as each expert in the course's reference model [§3]:

`FFN(x) = W_down( SiLU(W_gate x) ⊙ W_up x )`

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.gate = nn.Linear(d, hidden, bias=False)
        self.up = nn.Linear(d, hidden, bias=False)
        self.down = nn.Linear(hidden, d, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class Attention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.qkv, self.proj = h, nn.Linear(d, 3 * d), nn.Linear(d, d)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = (t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in self.qkv(x).split(C, 2))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(d), Attention(d, h)
        self.ln2, self.ffn = nn.LayerNorm(d), SwiGLU(d, ffn_mult * d)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        return x + self.drop(self.ffn(self.ln2(x)))

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab_size, n_embd), nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(Block(n_embd, n_head) for _ in range(n_layer))
        self.ln_f, self.head = nn.LayerNorm(n_embd), nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(lambda m: nn.init.normal_(m.weight, std=0.02) if isinstance(m, (nn.Linear, nn.Embedding)) else None)
    def forward(self, idx, targets=None):
        x = self.tok(idx) + self.pos(torch.arange(idx.size(1), device=idx.device))
        for b in self.blocks:
            x = b(x)
        logits = self.head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

## 4. The MoE layer [§7, §11, §13]

Per token:
1. **Score.** The router (fp32) gives one logit per expert, and sigmoid turns them into scores.
2. **Choose.** Add the balancing **bias**, then keep the top-k. Early in training this is sampled instead (probabilistic top-k).
3. **Weight.** Take the chosen experts' *raw* scores (no bias), rescale them to sum to 1, and multiply by the routed scaling factor.
4. **Combine.** `y = shared(x) + Σ wᵢ · Eᵢ(x)`

It is dropless: each expert simply processes however many tokens chose it.

In [ ]:
class MoE(nn.Module):
    def __init__(self, shared, experts, d, routed_scale=1.0):
        super().__init__()
        self.shared = shared                        # always on (or None)
        self.experts = nn.ModuleList(experts)
        self.E, self.k, self.routed_scale = len(experts), top_k, routed_scale
        self.router = nn.Linear(d, self.E, bias=False)
        nn.init.normal_(self.router.weight, std=router_init_std)
        self.register_buffer("bal_bias", torch.zeros(self.E))
        self.register_buffer("load_acc", torch.zeros(self.E))
        self.prob_select = False
        self.aux_loss = torch.zeros(())

    def forward(self, x):
        B, T, C = x.shape
        x = x.reshape(-1, C); N = x.size(0)
        logits = self.router(x.float())                           # router in fp32
        s = torch.sigmoid(logits) if score_fn == "sigmoid" else logits.softmax(-1)
        sel = s + self.bal_bias if balance == "lossfree" else s   # bias only decides WHO
        if self.training and self.prob_select:                    # probabilistic top-k
            p = sel.clamp_min(1e-6); p = p / p.sum(-1, keepdim=True)
            idx = torch.multinomial(p, self.k)
        else:                                                     # hard top-k
            idx = sel.topk(self.k, dim=-1).indices
        w = s.gather(-1, idx)                                     # HOW MUCH: raw scores
        w = (self.routed_scale * w / w.sum(-1, keepdim=True)).to(x.dtype)

        out = self.shared(x) if self.shared is not None else torch.zeros_like(x)
        flat = idx.reshape(-1)
        tok = torch.arange(N, device=x.device).repeat_interleave(self.k)
        fw = w.reshape(-1)
        for e, expert in enumerate(self.experts):
            m = flat == e
            if m.any():
                t = tok[m]
                out = out.index_add(0, t, fw[m, None] * expert(x[t]))

        load = torch.bincount(flat, minlength=self.E).float()     # counted over the whole batch [§14]
        if self.training:
            self.last_load = load
            self.load_acc += load

        s_norm = s / s.sum(-1, keepdim=True)
        if balance == "aux":                                      # Switch loss: N * sum f_i P_i [§12]
            f = load / (N * self.k)
            self.aux_loss = aux_alpha * self.E * (f * s_norm.mean(0)).sum()
        elif balance == "lossfree" and seq_aux_alpha > 0:         # tiny per-sequence loss [§13]
            counts = F.one_hot(idx.view(B, T * self.k), self.E).sum(1).float()     # (B, E)
            f = counts * self.E / (self.k * T)
            P = s_norm.view(B, T, self.E).mean(1)
            self.aux_loss = seq_aux_alpha * (f * P).sum(-1).mean()
        else:
            self.aux_loss = torch.zeros((), device=x.device)
        return out.reshape(B, T, C)

    @torch.no_grad()
    def update_bias(self):
        l = self.last_load                     # busier than average -> bias down, quieter -> bias up
        self.bal_bias += bias_gamma * torch.sign(l.mean() - l)

def moe_layers(model):
    return [m for m in model.modules() if isinstance(m, MoE)]

## 5. Growing: dense SwiGLU → shared expert + routed experts [§15]

With `use_shared = True` the dense network's inner neurons are split in half. The **shared expert** takes neurons `[0 : H/2]`, and the routed experts are built from the other half, which we call the *source*:

| `init_method` | Each routed expert starts as | Conversion |
|---|---|---|
| `copy` | the whole source (sparse upcycling) | exact, because the top-k weights sum to 1 |
| `partition` | an overlapping random `part_frac` slice of the source neurons; outputs × `1/part_frac` (routed scaling factor) | close; the experts start different |
| `drop` | the whole source with a fraction `drop_ratio` of its neurons redrawn | close; the experts start near each other but different |

With `use_shared = False` the source is the whole dense network.

In [ ]:
def swiglu_from(src, idx):
    new = SwiGLU(src.gate.in_features, len(idx)).to(src.gate.weight.device)
    with torch.no_grad():
        new.gate.weight.copy_(src.gate.weight[idx])
        new.up.weight.copy_(src.up.weight[idx])
        new.down.weight.copy_(src.down.weight[:, idx])
    return new

def make_experts(src, source_idx):
    Hs = len(source_idx)
    experts = []
    for _ in range(n_experts):
        if init_method == "partition":
            pick = source_idx[torch.randperm(Hs)[: int(part_frac * Hs)]]
            experts.append(swiglu_from(src, pick.sort().values))
            continue
        e = swiglu_from(src, source_idx)
        with torch.no_grad():
            if init_method == "drop":
                redraw = torch.randperm(Hs)[: int(drop_ratio * Hs)]
                e.gate.weight[redraw] = torch.randn_like(e.gate.weight[redraw]) * 0.02
                e.up.weight[redraw] = torch.randn_like(e.up.weight[redraw]) * 0.02
                e.down.weight[:, redraw] = 0.0 if zero_init_dropped else torch.randn_like(e.down.weight[:, redraw]) * 0.02
            elif copy_noise > 0:
                for p in e.parameters():
                    p.add_(copy_noise * p.std() * torch.randn_like(p))
        experts.append(e)
    return experts

def upcycle(dense_model):
    moe_model = copy.deepcopy(dense_model)
    for b in moe_model.blocks:
        src = b.ffn; H = src.gate.out_features
        if use_shared:
            shared = swiglu_from(src, torch.arange(H // 2))
            source_idx = torch.arange(H // 2, H)
        else:
            shared, source_idx = None, torch.arange(H)
        scale = 1 / part_frac if init_method == "partition" else 1.0
        b.ffn = MoE(shared, make_experts(src, source_idx), n_embd, routed_scale=scale).to(device)
    return moe_model

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    idle = sum((m.E - m.k) * sum(p.numel() for p in m.experts[0].parameters()) for m in moe_layers(model))
    return total, total - idle

## 6. Training utilities

In [ ]:
def balance_metrics(shares):
    # shares: (..., experts) fractions of routing decisions that sum to 1 over the last dim
    E = shares.shape[-1]
    maxvio = shares.max(-1).values * E - 1                         # 0 = even, 1 = busiest gets 2x average
    cv = shares.std(-1, unbiased=False) / shares.mean(-1)          # coefficient of variation, 0 = even
    p = shares.clamp_min(1e-12)
    eff = torch.exp(-(p * p.log()).sum(-1))                        # effective number of experts, E = even
    return maxvio, cv, eff

@torch.no_grad()
def eval_loss(model):
    model.eval()
    l = sum(model(x, y)[1].item() for x, y in val_set) / len(val_set)
    model.train()
    return l

def train(model, iters, lr, warm, step0, log, tag):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    moes = moe_layers(model)
    t0 = time.time()
    for it in range(iters + 1):
        if it % eval_every == 0 or it == iters:
            v = eval_loss(model); log["val"].append((step0 + it, v))
            msg = f"[{tag}] step {step0 + it:5d}  val {v:.4f}"
            if moes:
                shares = torch.stack([m.load_acc / m.load_acc.sum().clamp_min(1) for m in moes]).cpu()
                log["load"].append((step0 + it, shares))
                for m in moes: m.load_acc.zero_()
                if it > 0:
                    msg += f"  MaxVio {(shares.max(-1).values * n_experts - 1).mean():.2f}"
            print(msg + f"  ({time.time() - t0:.0f}s)")
        if it == iters:
            break
        for m in moes:
            m.prob_select = it < prob_topk_iters
        frac = it / max(1, iters)       # linear warmup, then cosine decay to 10%
        lr_t = lr * min(1.0, (it + 1) / warm) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * frac)))
        for g in opt.param_groups: g["lr"] = lr_t
        x, y = get_batch("train")
        _, loss = model(x, y)
        total = loss + sum(m.aux_loss for m in moes) if moes else loss
        opt.zero_grad(set_to_none=True)
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if balance == "lossfree":
            for m in moes: m.update_bias()
        log["train"].append((step0 + it, loss.item()))
    return model

@torch.no_grad()
def sample(model, n=300):
    model.eval()
    idx = torch.zeros((1, 1), dtype=torch.long, device=device)
    for _ in range(n):
        logits, _ = model(idx[:, -block_size:])
        idx = torch.cat([idx, torch.multinomial(logits[:, -1].softmax(-1), 1)], 1)
    model.train()
    return "".join(itos[i] for i in idx[0].tolist())

## 7. The dense model

Downloaded from the GitHub repo, so every seed starts from exactly the model in the README. In quick test mode, or if the download fails, a new dense model is trained instead.

In [ ]:
DENSE_URL = "https://raw.githubusercontent.com/kunal2016/mixture-of-experts/main/dense.pt"
dense = GPT().to(device)
loaded = False
if not QUICK:
    try:
        if not os.path.exists("dense.pt"):
            print("downloading", DENSE_URL)
            urllib.request.urlretrieve(DENSE_URL, "dense.pt")
        dense.load_state_dict(torch.load("dense.pt", map_location=device))
        loaded = True
    except Exception as e:
        print("could not load dense.pt:", e)
        if os.path.exists("dense.pt"):
            os.remove("dense.pt")                                 # a bad download is retried on the next run
if not loaded:
    print("training a new dense model")
    torch.manual_seed(1337)
    train(dense, dense_iters, lr_dense, warmup, 0, {"train": [], "val": [], "load": []}, "dense")
loss_before = eval_loss(dense)
print(f"dense model validation loss: {loss_before:.4f}" + ("   (the README run measured 0.7848)" if loaded else ""))

## 8. Settings

Results go to Google Drive (`MyDrive/moe_seeds/`) when you allow access, so they survive a disconnect. Otherwise they are saved next to the notebook; download them before the session ends.

In [ ]:
SEEDS = [0, 1, 2]
EXP_ITERS = 1000
SAVE_TO_DRIVE = True

VARIANTS = {   # name: settings that differ from section 1
    "drop (baseline)":     {},
    "copy":                {"init_method": "copy"},
    "partition":           {"init_method": "partition"},
    "copy, hard top-k":    {"init_method": "copy", "prob_topk_iters": 0},
    "no balancing":        {"balance": "none"},
    "aux loss":            {"balance": "aux"},
    "softmax router":      {"score_fn": "softmax"},
    "no shared expert":    {"use_shared": False},
    "16 experts, top-4":   {"init_method": "partition", "n_experts": 16, "top_k": 4, "part_frac": 0.25},
}
PANELS = {
    "Growing method":         ["drop (baseline)", "copy", "partition"],
    "Clone collapse":         ["copy", "copy, hard top-k"],
    "Balancing":              ["drop (baseline)", "no balancing", "aux loss"],
    "Router & shared expert": ["drop (baseline)", "softmax router", "no shared expert"],
    "Granularity":            ["partition", "16 experts, top-4"],
}
BASELINE = "drop (baseline)"
CONTROL = "dense, continued"
RUN = list(VARIANTS)

if QUICK:
    SEEDS, EXP_ITERS, SAVE_TO_DRIVE = [0, 1], 40, False

KNOBS = ["init_method", "prob_topk_iters", "balance", "score_fn", "use_shared", "n_experts", "top_k", "part_frac"]
DEFAULTS = {k: globals()[k] for k in KNOBS}

OUT_DIR = "."
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = "/content/drive/MyDrive/moe_seeds"
    except Exception as e:
        print("not saving to Drive:", e)
os.makedirs(os.path.join(OUT_DIR, "logs"), exist_ok=True)
RESULTS = os.path.join(OUT_DIR, "seeds_runs.csv")
print("results file:", RESULTS)

## 9. Run every variant with every seed

The loop goes seed by seed, so even a partly finished run gives complete, comparable sets. Each finished run is written to `seeds_runs.csv` straight away.

In [ ]:
import json
import pandas as pd

def load_stats(log):
    sh = torch.stack([s for _, s in log["load"]][1:])            # (evals, layers, experts)
    E = sh.shape[-1]
    mv, cv, eff = balance_metrics(sh)                            # each (evals, layers)
    dead = (sh < 0.1 / E).sum((-1, -2))
    return [s for s, _ in log["load"]][1:], mv.mean(-1), cv.mean(-1), eff.mean(-1), dead, E, sh.shape[1] * E

def save_log(name, seed, log, extra=None):
    out = {"val": log["val"]}
    if extra:
        out.update(extra)
    with open(os.path.join(OUT_DIR, "logs", f"{name.replace(' ', '_').replace(',', '')}_seed{seed}.json"), "w") as f:
        json.dump(out, f)

rows, done = [], set()
if os.path.exists(RESULTS):
    rows = pd.read_csv(RESULTS).to_dict("records")
    done = {(r["variant"], int(r["seed"])) for r in rows}
    print(f"resuming: {len(done)} runs already finished")

def record(row):
    rows.append(row)
    pd.DataFrame(rows).to_csv(RESULTS, index=False)

t_start = time.time()
for seed in SEEDS:
    if (CONTROL, seed) not in done:
        torch.manual_seed(seed)                                   # same batches as the variants for this seed
        ctrl = copy.deepcopy(dense)
        log = {"train": [], "val": [], "load": []}
        train(ctrl, EXP_ITERS, lr_cont, warmup, dense_iters, log, f"{CONTROL} | seed {seed}")
        record({"variant": CONTROL, "seed": seed, "final val loss": log["val"][-1][1]})
        save_log(CONTROL, seed, log)
        del ctrl
    for name in RUN:
        if (name, seed) in done:
            continue
        globals().update(DEFAULTS); globals().update(VARIANTS[name])
        torch.manual_seed(1000 + seed)                            # conversion randomness: redrawn neurons, router init
        m = upcycle(dense)
        after = eval_loss(m)
        tot, act = count_params(m)
        log = {"train": [], "val": [], "load": []}
        torch.manual_seed(seed)                                   # training randomness: batches, sampled routing
        train(m, EXP_ITERS, lr_cont, warmup, dense_iters, log, f"{name} | seed {seed}")
        steps, mv, cv, eff, dead, E, n_total = load_stats(log)
        record({
            "variant": name, "seed": seed,
            "total params (M)": round(tot / 1e6, 2), "active params (M)": round(act / 1e6, 2),
            "jump at conversion": after - loss_before,
            "final val loss": log["val"][-1][1],
            "MaxVio final": float(mv[-1]), "MaxVio peak": float(mv.max()),
            "CV final": float(cv[-1]),
            "effective experts final": float(eff[-1]), "experts per layer": E,
            "dead final": int(dead[-1]), "dead peak": int(dead.max()), "experts total": n_total,
        })
        save_log(name, seed, log, {"steps": steps, "maxvio": mv.tolist()})
        del m
        if device == "cuda":
            torch.cuda.empty_cache()
    print(f"--- seed {seed} finished, {(time.time() - t_start) / 60:.1f} min so far")
globals().update(DEFAULTS)

## 10. Results: mean ± spread across seeds

"vs dense continued" is computed **seed by seed** (each variant against the dense control trained on the same batches), then averaged.

In [ ]:
df = pd.read_csv(RESULTS)
ctrl_loss = df[df.variant == CONTROL].set_index("seed")["final val loss"]
df["vs dense continued"] = df["final val loss"] - df["seed"].map(ctrl_loss)

def pm(series, digits):
    s = series.dropna()
    if len(s) == 0:
        return "—"
    if len(s) == 1:
        return f"{s.iloc[0]:.{digits}f}"
    return f"{s.mean():.{digits}f} ± {s.std(ddof=1):.{digits}f}"

order = [CONTROL] + [v for v in RUN if v in set(df.variant)]
COLS = [("jump at conversion", 4), ("final val loss", 4), ("vs dense continued", 4),
        ("MaxVio final", 3), ("MaxVio peak", 3), ("CV final", 3),
        ("effective experts final", 2), ("dead final", 1), ("dead peak", 1)]
table = []
for v in order:
    g = df[df.variant == v]
    row = {"variant": v, "seeds": len(g)}
    if v != CONTROL:
        row["total / active (M)"] = f"{g['total params (M)'].iloc[0]:.2f} / {g['active params (M)'].iloc[0]:.2f}"
        row["experts per layer"] = int(g["experts per layer"].iloc[0])
    for c, d in COLS:
        row[c] = pm(g[c], d) if c in g else "—"
    table.append(row)
summary = pd.DataFrame(table).set_index("variant").fillna("—")
summary.to_csv(os.path.join(OUT_DIR, "seeds_summary.csv"))
summary

### Which differences are real?

Each comparison is **paired by seed**: for every seed, take the variant's value minus the reference's value (same training batches), then test whether the average difference is far enough from zero given how much it varies between seeds (paired t-test, two-sided). **real** means p < 0.05; **within noise** means the seeds can't separate them.

With only 3 seeds the test is strict: small real effects may still show as "within noise". A "real" verdict, on the other hand, is trustworthy.

In [ ]:
from scipy import stats

def paired(v, ref, metric):
    a = df[df.variant == v].set_index("seed")[metric]
    b = df[df.variant == ref].set_index("seed")[metric]
    common = a.index.intersection(b.index)
    d = (a[common] - b[common]).dropna()
    if len(d) < 2:
        return f"{d.mean():+.4f}" if len(d) else "—", "need ≥ 2 seeds"
    mean, sd = d.mean(), d.std(ddof=1)
    if sd == 0:
        p = 0.0 if mean != 0 else 1.0
    else:
        p = 2 * stats.t.sf(abs(mean) / (sd / len(d) ** 0.5), len(d) - 1)
    return f"{mean:+.4f} ± {sd:.4f}", ("real" if p < 0.05 else "within noise") + f"  (p = {p:.3f})"

checks = []
for v in [v for v in RUN if v in set(df.variant)]:
    r = {"variant": v}
    r["Δ val loss vs dense continued"], r["verdict (vs dense)"] = paired(v, CONTROL, "final val loss")
    if v != BASELINE:
        r["Δ val loss vs baseline"], r["verdict (loss)"] = paired(v, BASELINE, "final val loss")
        r["Δ MaxVio vs baseline"], r["verdict (MaxVio)"] = paired(v, BASELINE, "MaxVio final")
    checks.append(r)
verdicts = pd.DataFrame(checks).set_index("variant").fillna("—")
verdicts.to_csv(os.path.join(OUT_DIR, "seeds_verdicts.csv"))
verdicts

### Curves: mean across seeds, shaded from the lowest to the highest seed

In [ ]:
BLUE, ORANGE, GREEN, YELLOW, GRID, INK = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#d9d9d4", "#3d3d3a"
SLOTS = [BLUE, ORANGE, GREEN, YELLOW]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GRID,
                     "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "font.size": 10})

def curves(name, key):
    runs = []
    for s in SEEDS:
        p = os.path.join(OUT_DIR, "logs", f"{name.replace(' ', '_').replace(',', '')}_seed{s}.json")
        if os.path.exists(p):
            d = json.load(open(p))
            if key == "val":
                runs.append(([x for x, _ in d["val"]], [y for _, y in d["val"]]))
            elif "maxvio" in d:
                runs.append((d["steps"], d["maxvio"]))
    if not runs:
        return None
    n = min(len(r[1]) for r in runs)
    xs = runs[0][0][:n]
    ys = torch.tensor([r[1][:n] for r in runs])
    return xs, ys.mean(0), ys.min(0).values, ys.max(0).values

panels = {p: [v for v in vs if v in set(df.variant)] for p, vs in PANELS.items()}
panels = {p: vs for p, vs in panels.items() if vs}
P = len(panels)
fig, axes = plt.subplots(2, P, figsize=(3.6 * P, 6.4), squeeze=False)
for j, (title, vs) in enumerate(panels.items()):
    ax_l, ax_m = axes[0, j], axes[1, j]
    c = curves(CONTROL, "val")
    if c:
        ax_l.plot(c[0], c[1], color=INK, lw=1.4, ls="--", label=CONTROL)
    for i, v in enumerate(vs):
        col = SLOTS[i % len(SLOTS)]
        c = curves(v, "val")
        if c:
            ax_l.plot(c[0], c[1], color=col, lw=2, label=v)
            ax_l.fill_between(c[0], c[2], c[3], color=col, alpha=0.15, lw=0)
        c = curves(v, "maxvio")
        if c:
            ax_m.plot(c[0], c[1], color=col, lw=2, label=v)
            ax_m.fill_between(c[0], c[2], c[3], color=col, alpha=0.15, lw=0)
    ax_l.set_title(title)
    ax_l.legend(frameon=False, fontsize=8)
    ax_m.set_ylim(bottom=0)
    for ax in (ax_l, ax_m):
        ax.grid(axis="y", color=GRID, lw=0.6)
    ax_m.set_xlabel("step")
axes[0, 0].set_ylabel("validation loss")
axes[1, 0].set_ylabel("MaxVio (mean over layers)\n0 = even, 1 = 2× average")
fig.suptitle(f"Mean of {len(SEEDS)} seeds; shading spans the lowest to the highest seed", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "seeds_experiments.png"), dpi=150, bbox_inches="tight")
plt.show()

### Files to keep

All in `OUT_DIR` (printed in section 8):

* `seeds_runs.csv`: one row per run (variant × seed)
* `seeds_summary.csv`: the mean ± spread table
* `seeds_verdicts.csv`: the "which differences are real" table
* `seeds_experiments.png`: the chart
* `logs/`: per-run curves, used to redraw the chart after a resume

Download them, or copy them from Drive, into the repo folder. Then paste the two tables here to update the README.